# Amazon.sa Price Tracker

Scrapes a product page on Amazon.sa with **BeautifulSoup** + **Requests**, then logs the price, rating, and brand to a CSV so the price can be tracked over time.

**Features**
- Prompts you for the Amazon.sa product URL you want to track
- Pulls title, price, brand, and rating from that product page
- Appends each check to a CSV (keeps price history instead of overwriting it)
- Configurable polling interval for periodic checks

**Known limitations**
- Amazon.sa occasionally serves a bot-detection / CAPTCHA page instead of the real listing; a custom `User-Agent` header helps but isn't a full workaround
- Email alerting (via `smtplib`) is scaffolded but not yet wired into the price-check flow
- No retry/backoff logic yet on failed requests


In [ ]:
from bs4 import BeautifulSoup
import requests
import pandas as pd
import datetime
import time
import csv
import os
import smtplib  # reserved for a future email-alert feature (not yet wired in)


In [ ]:
# ---- Configuration ----

PRODUCT_URL = input("Enter the Amazon.sa product URL to track: ").strip()
CSV_FILE = "AmazonWebScraperDataset.csv"
CHECK_INTERVAL_SECONDS = 60 * 60 * 24  # 24 hours between checks

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/78.0.3904.108 Safari/537.36",
    "Accept-Encoding": "gzip, deflate",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "DNT": "1",
    "Connection": "close",
    "Upgrade-Insecure-Requests": "1",
}


In [ ]:
def scrape_product(url: str) -> dict | None:
    """Fetch a single Amazon.sa product page and pull out title, price, brand, and rating.

    Returns None if the page doesn't match the expected structure (e.g. a
    bot-detection page or a layout change), instead of raising.
    """
    page = requests.get(url, headers=HEADERS)
    soup = BeautifulSoup(page.content, "html.parser")

    title_tag = soup.find(id="productTitle")
    price_tag = soup.select_one("span.a-price span.a-offscreen")
    brand_tag = soup.find(id="bylineInfo")
    rating_tag = soup.find("span", {"class": "a-icon-alt"})

    if not all([title_tag, price_tag, brand_tag, rating_tag]):
        print("Could not parse product page (possible bot-detection page or layout change).")
        return None

    return {
        "Title": title_tag.get_text(strip=True),
        "Price": price_tag.get_text(strip=True),
        "Brand": brand_tag.get_text(strip=True),
        "Rating": rating_tag.get_text(strip=True),
        "Date": datetime.date.today().isoformat(),
    }


In [ ]:
def log_to_csv(row: dict, filename: str = CSV_FILE):
    """Append one row to the CSV, writing the header only the first time.

    (Earlier version opened the file with mode "w" inside the polling loop,
    which overwrote the whole file on every check instead of building up a
    price history -- fixed here to open in append mode.)
    """
    file_exists = os.path.isfile(filename)
    with open(filename, "a", newline="", encoding="UTF8") as f:
        writer = csv.DictWriter(f, fieldnames=row.keys())
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)


In [ ]:
def check_price():
    row = scrape_product(PRODUCT_URL)
    if row:
        log_to_csv(row)
        print(f"{row['Date']}  {row['Price']}  —  {row['Title']}")


## Run once

In [ ]:
check_price()


## Run continuously
Polls every `CHECK_INTERVAL_SECONDS` until interrupted.

In [ ]:
try:
    while True:
        check_price()
        time.sleep(CHECK_INTERVAL_SECONDS)
except KeyboardInterrupt:
    print("Stopped by user.")


## Inspect collected data

In [ ]:
df = pd.read_csv(CSV_FILE)
df
